<a href="https://colab.research.google.com/github/udlbook/udlbook/blob/main/Notebooks/Chap08/8_4_High_Dimensional_Spaces.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Notebook 8.4: High-dimensional spaces**

This notebook investigates the strange properties of high-dimensional spaces as discussed in the notes at the end of chapter 8.

Work through the cells below, running each cell in turn. In various places you will see the words "TODO". Follow the instructions at these places and make predictions about what is going to happen or write code to complete the functions.

Contact me at udlbookmail@gmail.com if you find any mistakes or have any suggestions.

In [36]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.special as sci

# How close are points in high dimensions?

In this part of the notebook, we investigate how close random points are in 2D, 100D, and 1000D.   In each case, we generate 1000 points and calculate the Euclidean distance between each pair.  

In [37]:
# Fix the random seed so we all have the same random numbers
np.random.seed(0)
n_data = 1000
# Create 1000 data examples (columns) each with 2 dimensions (rows)
n_dim = 2
x_2D = np.random.normal(size=(n_dim,n_data))
# Create 1000 data examples (columns) each with 100 dimensions (rows)
n_dim = 100
x_100D = np.random.normal(size=(n_dim,n_data))
# Create 1000 data examples (columns) each with 1000 dimensions (rows)
n_dim = 1000
x_1000D = np.random.normal(size=(n_dim,n_data))

In [38]:
(np.expand_dims(x_1000D,axis=2) - np.expand_dims(x_1000D,axis=1)).shape

(1000, 1000, 1000)

In [40]:
def distance_ratio(x):
  # TODO -- replace the two lines below to calculate the largest and smallest Euclidean distance between
  # the data points in the columns of x.  DO NOT include the distance between the data point
  # and itself (which is obviously zero)
  smallest_dist = 1.0
  largest_dist = 1.0

  distances = np.linalg.norm(np.expand_dims(x,axis=2) - np.expand_dims(x,axis=1),axis=0)
  # Keep the strict upper triangle: every pair counted once, and no point paired with itself
  distances = distances[np.triu_indices(distances.shape[0], k=1)]
  sorted_distances = np.sort(distances)
  largest_dist = sorted_distances[-1]
  smallest_dist = sorted_distances[0]
  # Calculate the ratio and return
  dist_ratio = largest_dist / smallest_dist
  return dist_ratio

In [41]:
print('Ratio of largest to smallest distance 2D: %3.3f'%(distance_ratio(x_2D)))
print('Ratio of largest to smallest distance 100D: %3.3f'%(distance_ratio(x_100D)))
print('Ratio of largest to smallest distance 1000D: %3.3f'%(distance_ratio(x_1000D)))


Ratio of largest to smallest distance 2D: 2840.258
Ratio of largest to smallest distance 100D: 2.038
Ratio of largest to smallest distance 1000D: 1.221


If you did this right, you will see that the distance between the nearest and farthest two points in high dimensions is almost the same.  


My answer: with 1000 dimensions the distance between the farthest pair of points and the closest pair is almost the same. Each point is defined by 1000 coordinates, and the distance between two points is the length of the vector we get by subtracting one point's coordinates from the other's.

The reason the largest and smallest distances converge is averaging. The squared distance is a sum of 1000 independent per-coordinate contributions, and when we add up that many independent terms the total's mean grows like the number of dimensions while its spread only grows like the square root of it. In practice the typical distance grows like sqrt(2D), reaching about 44.7 at 1000 dimensions, while the spread of distances stays at roughly 1 in every dimension. For two points to be unusually close they would have to be close in all 1000 coordinates at once, which essentially never happens, and any single coordinate where they differ wildly is diluted by the other 999.

So it is not that the points are incomparable. They all sit at almost exactly the same distance from each other, which means the comparison stops telling us anything: nearest and farthest become indistinguishable.

# Volume of a hypersphere

In the second part of this notebook we calculate the volume of a hypersphere of radius 0.5 (i.e., of diameter 1) as a function of the radius.  Note that you can check your answer by doing the calculation for 2D using the standard formula for the area of a circle and making sure it matches.

In [42]:
def volume_of_hypersphere(diameter, dimensions):
  # Formula given in Problem 8.7 of the book
  # You will need sci.gamma()
  # Check out:    https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.gamma.html
  # Also use this value for pi
  pi = np.pi
  radius = diameter/2

  volume = (radius**dimensions * pi**(dimensions/2)) / sci.gamma((dimensions/2) + 1)


  return volume


In [43]:
diameter = 1.0
for c_dim in range(1,11):
  print("Volume of unit diameter hypersphere in %d dimensions is %3.3f"%(c_dim, volume_of_hypersphere(diameter, c_dim)))

Volume of unit diameter hypersphere in 1 dimensions is 1.000
Volume of unit diameter hypersphere in 2 dimensions is 0.785
Volume of unit diameter hypersphere in 3 dimensions is 0.524
Volume of unit diameter hypersphere in 4 dimensions is 0.308
Volume of unit diameter hypersphere in 5 dimensions is 0.164
Volume of unit diameter hypersphere in 6 dimensions is 0.081
Volume of unit diameter hypersphere in 7 dimensions is 0.037
Volume of unit diameter hypersphere in 8 dimensions is 0.016
Volume of unit diameter hypersphere in 9 dimensions is 0.006
Volume of unit diameter hypersphere in 10 dimensions is 0.002


Note: With 1000 dimensions the hypersphere takes up almost no volume. Picture it inscribed in a cube of side 1, a cube whose volume is 1 in every dimension, and the fraction it fills shrinks toward zero as dimensions increase.

Conceptually the high-dimensional cube is a sea urchin. The body is the hypersphere and the spikes are its 2^D corners. Almost all the volume is in the spikes, so a randomly sampled point almost certainly lands in one.

What that means: landing inside the ball would require being un-extreme in every coordinate simultaneously. That conjunction becomes vanishingly unlikely as dimensions grow. Every point is unusual in some combination of coordinates, and each sits in its own spike.

This does not make the points incomparable. The opposite is true. Since every point is out on some spike and all spikes are about equally far apart, every pair ends up at nearly the same distance. The comparison stops discriminating: nearest and farthest become indistinguishable, which is exactly what part 1 measured.

That is why "the network interpolates between nearby training points" is a shaky story here. No training point is meaningfully nearer to a test point than any other. What rescues it in practice is that real data has correlated coordinates, so it occupies a low-dimensional surface rather than spreading into all the corners.


The escape from all of this is **intrinsic dimensionality**. Everything above assumes the coordinates are independent, which is what np.random.normal gives us and which is the worst possible case. What actually governs the behaviour is not how many coordinates you store (the ambient dimension) but how many numbers you would genuinely need to describe the data (the intrinsic dimension). If the coordinates are correlated, the data does not spread into the corners at all. It lies on a lower-dimensional surface inside the cube, and it suffers only the curse of that surface's dimension.

This is easy to check. Take 1000 points with 50 coordinates generated from just 2 underlying factors. They have exactly as many coordinates as the independent case, but their largest-to-smallest distance ratio comes out around 1800 rather than 2.5. They behave like 2D data, because that is what they are underneath. Those 50 coordinates are 2 numbers in disguise.

Real data is overwhelmingly of this kind. A 224x224 colour photograph is 150,528 numbers, but filling those with random values gives television static every time and never a face. Photographs occupy a vanishingly thin, curved sheet inside that space, because neighbouring pixels are correlated, lighting is smooth, and objects are coherent. Its intrinsic dimension is perhaps dozens.

This is the manifold hypothesis, and it is the resolution to the whole chapter. The notebook shows the worst case in order to break our low-dimensional intuitions. Deep learning still works because real data never looks like np.random.normal, so nearest neighbours and interpolation continue to mean something for faces and sentences even though they would mean nothing for points scattered through the corners of a 150,000-dimensional cube.



You should see that the volume decreases to almost nothing in high dimensions.  All of the volume is in the corners of the unit hypercube (which always has volume 1).

# Proportion of hypersphere in outer shell

In the third part of the notebook you will calculate what proportion of the volume of a hypersphere is in the outer 1% of the radius/diameter.  Calculate the volume of a hypersphere and then the volume of a hypersphere with 0.99 of the radius and then figure out the ratio.  

In [48]:
def get_prop_of_volume_in_outer_1_percent(dimension):
  # TODO -- replace this line

  
  proportion = 1.0

  pi = np.pi
  radius = diameter/2
  volume1 = (radius**dimension * pi**(dimension/2)) / sci.gamma((dimension/2) + 1)
  volume2 = ((radius*0.99)**dimension * pi**(dimension/2)) / sci.gamma((dimension/2) + 1)

  proportion = (volume1 - volume2) / volume1


  return proportion

In [49]:
# While we're here, let's look at how much of the volume is in the outer 1% of the radius
for c_dim in [1,2,10,20,50,100,150,200,250,300]:
  print('Proportion of volume in outer 1 percent of radius in %d dimensions =%3.3f'%(c_dim, get_prop_of_volume_in_outer_1_percent(c_dim)))

Proportion of volume in outer 1 percent of radius in 1 dimensions =0.010
Proportion of volume in outer 1 percent of radius in 2 dimensions =0.020
Proportion of volume in outer 1 percent of radius in 10 dimensions =0.096
Proportion of volume in outer 1 percent of radius in 20 dimensions =0.182
Proportion of volume in outer 1 percent of radius in 50 dimensions =0.395
Proportion of volume in outer 1 percent of radius in 100 dimensions =0.634
Proportion of volume in outer 1 percent of radius in 150 dimensions =0.779
Proportion of volume in outer 1 percent of radius in 200 dimensions =0.866
Proportion of volume in outer 1 percent of radius in 250 dimensions =0.919
Proportion of volume in outer 1 percent of radius in 300 dimensions =0.951


You should see that by the time we get to 300 dimensions most of the volume is in the outer 1 percent. 

Note: This again shows that general intuition around ML is a bit wrong. In high-dimensional input spaces we are never in the centre. Essentially all of the volume lies near the boundary of whatever region we are in, so there is no meaningful "in between" for observed data. Averaging two real points lands at roughly 71% of the typical radius, which in 1000 dimensions is about 13 standard deviations inside the shell where the data actually sits, a place nothing occupies.

This is the same concentration effect as part 1 seen from another angle: everything on a thin shell, mutually near-orthogonal, therefore all pairs equidistant and boringly comparable. Together they break the idea that a network interpolates between nearby training points, since in the raw high-dimensional space no training point is meaningfully nearer than any other and the space between them is empty. As in part 2, what rescues this for real data is its low intrinsic dimensionality.
<br><br>

The conclusion of all of this is that in high dimensions you should be sceptical of your intuitions about how things work.  I have tried to visualize many things in one or two dimensions in the book, but you should also be sceptical about these visualizations!